# Detector de GoondVR en Google Colab (para pruebas cortas)

Pone en marcha el **mismo detector** que corre en tu PC (`detector/server.py`) dentro de Colab y lo abre a internet con
un túnel de Cloudflare (`trycloudflare.com`). GoondVR le envía las capturas y recibe qué hay en ellas. Al final de la
celda 4 salen **la dirección y la clave (token)** que pegas en GoondVR: *Ajustes → Image detection → On Google Colab*.

**Léelo antes de usarlo:**
- **Recomendación:** usa Colab para **entrenar** (cuaderno `train_detector.ipynb`) y tu PC para **detectar**. Este cuaderno es para probar un rato.
- Las preguntas frecuentes de Colab (https://research.google.com/colaboratory/faq.html) prohíben usos como "file hosting, media serving, or other web service offerings not related to interactive compute with Colab", "connecting to remote proxies" y, en las sesiones gratuitas, "bypassing the notebook UI to interact primarily via a web UI". Un servidor con túnel se parece a eso: **existe el riesgo de que Google limite o cierre la sesión**. No lo uses para dejarlo funcionando días.
- Las sesiones gratuitas duran como mucho unas 12 horas y se cierran si hay inactividad; al cerrarse, la dirección deja de funcionar y la próxima vez será **otra**.
- **Privacidad:** las capturas de tus canales salen de tu PC hacia Google (y pasan por Cloudflare). Si no quieres eso, usa el detector local.
- La clave se genera al azar en cada ejecución: sin ella nadie puede usar tu detector.
- Este cuaderno se escribió sin poder ejecutarlo en Colab: si una celda falla, el error dice cuál.

In [ ]:
#@title 1. Ajustes { display-mode: "form" }
MODELOS = "ninguno" #@param ["ninguno", "drive", "subir"]
CARPETA_MODELOS_EN_DRIVE = "/content/drive/MyDrive/goondvr/models" #@param {type:"string"}
DISPOSITIVO = "auto" #@param ["auto", "cpu", "gpu"]
# "ninguno": sin modelos tuyos (solo el modelo base). Los modelos que tengas en tu PC no están aquí: súbelos.
# "drive": una carpeta de Drive con una subcarpeta por modelo (<nombre>/meta.json, labels.txt, ...).
# "subir": un .zip con esas subcarpetas, que subes a mano.

In [ ]:
#@title 2. Instalar (1-2 minutos)
import os, shutil, subprocess, sys
!pip install -q numpy Pillow onnx
# con tarjeta gráfica; si no se puede, la versión de CPU (el detector elige solo la más rápida que dé los mismos resultados)
!pip install -q onnxruntime-gpu || pip install -q onnxruntime
!([ -d /content/goondvr ] || git clone -q --depth 1 --branch community-modifications https://github.com/Danyel2505/goondvr.git /content/goondvr)
MODELS = "/content/models"
os.makedirs(MODELS, exist_ok=True)
!cd /content/goondvr/detector && python fetch_models.py --models {MODELS}

In [ ]:
#@title 3. Tus modelos
import glob, zipfile
if MODELOS == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    for d in glob.glob(CARPETA_MODELOS_EN_DRIVE + "/*/meta.json"):
        nombre = os.path.basename(os.path.dirname(d))
        shutil.copytree(os.path.dirname(d), f"{MODELS}/{nombre}", dirs_exist_ok=True)
elif MODELOS == "subir":
    from google.colab import files
    for nombre in files.upload():
        with zipfile.ZipFile(nombre) as z:
            z.extractall(MODELS)
print("modelos:", sorted(d for d in os.listdir(MODELS) if os.path.exists(f"{MODELS}/{d}/meta.json")) or "ninguno (solo el modelo base)")

In [ ]:
#@title 4. Arrancar el detector y el túnel (esta celda debe quedarse ejecutando; para parar, deténla)
import re, secrets, threading, time, urllib.request
TOKEN = secrets.token_urlsafe(24)
env = dict(os.environ, DETECTOR_TOKEN=TOKEN)   # por variable de entorno: no aparece en la lista de procesos
srv = subprocess.Popen([sys.executable, "server.py", "--host", "127.0.0.1", "--port", "8099", "--models", MODELS,
                        "--device", DISPOSITIVO], cwd="/content/goondvr/detector", env=env)
!([ -x /content/cloudflared ] || (wget -q -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /content/cloudflared))
tun = subprocess.Popen(["/content/cloudflared", "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:8099"],
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
URL = None
t0 = time.time()
for linea in tun.stdout:
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", linea)
    if m:
        URL = m.group(0)
        break
    if time.time() - t0 > 90:
        break
if not URL:
    srv.terminate(); tun.terminate()
    raise RuntimeError("El túnel no dio dirección en 90 s: vuelve a ejecutar la celda")
threading.Thread(target=lambda: [None for _ in tun.stdout], daemon=True).start()   # vacía el registro del túnel
time.sleep(3)
try:
    ok = urllib.request.urlopen(URL + "/health", timeout=20).status == 200
except Exception as e:
    ok = f"no responde todavía ({e}); espera unos segundos y pulsa «Test connection» en GoondVR"
print("Dirección:", URL)
print("Token:    ", TOKEN)
print("Comprobación:", ok)
print("\nEn GoondVR: Ajustes → Image detection → «On Google Colab», pega la dirección y el token y pulsa «Test connection».")
try:
    while srv.poll() is None and tun.poll() is None:
        time.sleep(30)
    print("Algo se paró: vuelve a ejecutar la celda (la dirección nueva será distinta)")
finally:
    srv.terminate(); tun.terminate()